### Initialize the Environment:
##### Virtual Environment Commands

| Command | Linux/Mac | GitBash |
| ------- | --------- | ------- |
| Create | `python3 -m venv venv` | `python -m venv venv` |
| Activate | `source venv/bin/activate` | `source venv/Scripts/activate` |
| Install | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| Deactivate | `deactivate` | `deactivate` |
##### Select the Kernel (This will be in the Requirements.txt eventually)

Using the venv (Python 3.13.2) located  in venv/bin/python)

### **Analysis of the Data**
Capstone Project for Code:You Data Analysis track. This project analyzes Beer Recipes for frequency of uploads for various beer styles, while capturing preferences of strength, hopiness and batch size.    The goal of the project is to demonstrate a general knowledge of Python (Pandas, Numpy, MatLibPlot, Plotly), SQL(MySQL), Tableu, Cursor and ChatGPT.


In [ ]:
# pip install openpyxl
# %pip install matplotlib
# %pip install prettytable
# %pip install seaborn
# %pip install wordcloud

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import random
import numpy as np
import rich as rch
import seaborn as sns
from wordcloud import WordCloud
from rich.console import Console
from rich.table import Table


In [ ]:
# Un-pickle the dataframes from Beer_capstone.ipynb
beer_recipes = pd.read_pickle("cleaned_beer_recipes.pkl")
bjcp_style = pd.read_pickle("cleaned_styles_2021.pkl")

In [ ]:
# Confirm that data successfully transferred from Beer_capstone.ipynb
# print(beer_recipes.info)
# print(bjcp_style.info)

In [ ]:
beer_recipes.info()


### **Data Visualizations**
#### Load Functions for visualizations

Chart 1: The distribution of the Beer Recipes  among the 116 Styles. *Highlighting the top 2 styles because of their dominance*.

In [ ]:
# Function to plot the number of recipes per beer style
def total_occurrence(data: pd.DataFrame) -> None:
    # Group by 'Style' and count the number of occurrences
    style_counts = data['Style'].value_counts()

    # Sort the data in descending order
    style_counts = style_counts.sort_values(ascending=False)
    
    # Plot the bar chart horizontally
    fig, ax = plt.subplots(figsize=(20, 20))
    bars = ax.barh(style_counts.index, style_counts.values, color='orange') # type: ignore

    # Highlight the top 2 bars
    bars[0].set_color('red')
    bars[1].set_color('red')

    plt.xlabel('Number of Recipes')
    plt.ylabel('Beer Styles')
    plt.title('Number of Recipes per Beer Style')
    # Reverse the y-axis to have the largest bars at the top
    ax.invert_yaxis()
    plt.show()



Chart 2: Word Cloud frequency of Beer Recipes of Styles. *An alternate representative of frequency of styles of beer.* 

In [ ]:
# Function to create a word cloud of the frequency of beer styles
def create_beer_wordcloud(data: pd.DataFrame) -> None:
    # Create dictionary of style frequencies
    style_freq = data['Style'].value_counts().to_dict()
    
    # Create and generate a word cloud image
    wordcloud = WordCloud(
        width=1600, 
        height=800,
        background_color='black',
        colormap='YlOrRd',  # Yellow-Orange-Red color scheme
        min_font_size=10,
        max_font_size=150
    ).generate_from_frequencies(style_freq)
    
    # Display the word cloud
    plt.figure(figsize=(16,8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Beer Style Frequency Word Cloud', fontsize=20, pad=20)
    plt.show()



Chart 3: The **Top 50** beer styles in a vertical bar chart. *The rest of the styles were combined in an 'Other' category.*

In [ ]:
# Function to plot the number of recipes per beer style for the top 50 styles
def total_occurrence_abbrev_v(data: pd.DataFrame) -> None:
    # Group by 'Style' and count the number of occurrences
    style_counts = data['Style'].value_counts()

    # Sort the data in descending order
    style_counts = style_counts.sort_values(ascending=False)

    # Group the first 50 styles and the rest into "Other Style"
    top_styles = style_counts[:50]
    other_styles = pd.Series([style_counts[50:].sum()], index=['Other Style'])
    style_counts = pd.concat([top_styles, other_styles])

    
    # Plot the bar chart vertically
    fig, ax = plt.subplots(figsize=(16,8))
    bars = ax.bar(style_counts.index, style_counts.values, color='orange')  # type: ignore # Vertical bars

    # Rotate and align the tick labels so they look better
    plt.xticks(rotation=45, ha='right')  # Angle the category labels at 45 degrees

    # Highlight the top 2 bars
    bars[-50].set_color('red')  # Largest value (now at the left)
    bars[-51].set_color('red')

    # Formatting
    plt.ylabel('Number of Recipes')
    plt.xlabel('Beer Styles')
    plt.title('Number of Recipes per Beer Style')

    # Rotate and align the tick labels so they look better
    plt.xticks(rotation=45, ha='right')  # Angle the category labels at 45 degrees

    plt.show()


Chart 4: Number of Recipes per Category in a vertical bar chart.

In [ ]:
# Function to plot the number of recipes per beer category
def plot_category_counts(data: pd.DataFrame) -> None:
 
    # Group by 'Category' and count the number of occurrences
    category_counts = data['Category'].value_counts()
    
    # Plot the bar chart
    fig, ax = plt.subplots(figsize=(16,8))
    bars = ax.bar(category_counts.index, category_counts.values, color='tomato')
    
    # Set labels and title
    plt.xlabel('Beer Categories')
    plt.ylabel('Number of Recipes')
    plt.title('Number of Recipes per Beer Category')

    # Rotate and align the tick labels so they look better
    plt.xticks(rotation=45, ha='right')  # Angle the category labels at 45 degrees
    
    
    plt.show()


Visualization 5: Recipes by Category, segregated by Style in a stacked bar chart.

In [ ]:
# Function to plot the distribution of styles within each beer category
def plot_category_style_distribution(data: pd.DataFrame) -> None:

    # Group by 'Category' and 'Style' and count the number of occurrences
    category_style_counts = data.groupby(['Category', 'Style']).size().unstack().fillna(0)

    # Sort categories by total count (largest first)
    category_style_counts = category_style_counts.loc[category_style_counts.sum(axis=1).sort_values(ascending=False).index]

    # Generate a list of random colors
    random.seed(13)
    n_colors = len(category_style_counts.columns)
    colors = [mcolors.rgb2hex(plt.cm.hsv(i)) for i in random.sample(range(256), n_colors)]

    # Create a custom colormap
    custom_cmap = mcolors.ListedColormap(colors)

    # Plot the stacked bar chart
    fig, ax = plt.subplots(figsize=(16, 8))
    category_style_counts.plot(kind='bar', stacked=True, ax=ax, colormap=custom_cmap)

    # Add text box explanation
    plt.text(0.08, 0.95, 'Numbers above bars indicate\nthe count of unique styles\nin each category',
                transform=plt.gca().transAxes,  # Use axis coordinates
                bbox=dict(facecolor='white', 
                        edgecolor='black',
                        alpha=0.7,
                        boxstyle='round,pad=0.5'),
                fontsize=10,
                verticalalignment='top')

    plt.xlabel('Beer Categories', fontsize=14, labelpad=-20)
    plt.ylabel('Number of Recipes')
    plt.title('Number of Recipes per Beer Category and Style')

    # Rotate and align the tick labels so they look better
    plt.xticks(rotation=45, ha='right')  # Angle the category labels at 45 degrees

    # Count unique styles per category
    unique_styles_per_category = beer_recipes.groupby('Category')['Style'].nunique()

    # Add text annotations above bars
    for i, category in enumerate(category_style_counts.index):
        ax.text(i, category_style_counts.loc[category].sum(), f"{int(unique_styles_per_category.loc[category])}", ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Update legend with alphabetical order
    handles, labels = ax.get_legend_handles_labels()
    sorted_handles_labels = sorted(zip(handles, labels), key=lambda x: x[1])
    handles, labels = zip(*sorted_handles_labels)
    ax.legend(handles, labels, 
                title='Styles', 
                bbox_to_anchor=(0.5, -0.29),  # Move legend down
                loc='upper center', 
                ncol=7,  # Increase number of columns
                columnspacing=1.0,  # Reduce space between columns
                handletextpad=0.5,  # Reduce space between handle and text
                fontsize=8)  # Reduce font size

    plt.show()


Visualization 6: Recipe's and their adherence to the Style Guidelines presented in a table

In [ ]:
# Function to check if the recipe's Style is in compliance with the Style Guidelines by comparing the recipe's OG, FG, 
# ABV and IBU to the Style Guidelines. This function was working and during the cleanup process I mangled it. I used Cursor
# to fix it because I was running out of time.

def create_and_display_compliance_summary(data: pd.DataFrame, data2: pd.DataFrame) -> pd.DataFrame:
    """
    Create a comprehensive compliance summary table for beer recipes and display it.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame containing beer recipe data.
    data2 : pd.DataFrame
        DataFrame containing beer style guidelines.
    
    Returns:
    --------
    pd.DataFrame
        Summary table with compliance statistics for each beer style.
    """
    # First part: Check recipe compliance
    def check_recipe_compliance(recipes, guidelines):
        # Remove styles with missing FG values from bjcp_style. Missing FG values indicate the rest of the values don't exist.
        valid_styles = guidelines.dropna(subset=['fgmin', 'fgmax'])
        
        # Initialize results dictionary
        results = []
        
        # Process only recipes that have a style in our valid_styles
        valid_recipes = recipes[recipes['Style'].isin(valid_styles['Style'])]
        
        for _, recipe in valid_recipes.iterrows():
            style_guide = valid_styles[valid_styles['Style'] == recipe['Style']].iloc[0]
            
            # Check each requirement
            og_compliant = style_guide['ogmin'] <= recipe['OG'] <= style_guide['ogmax']
            fg_compliant = style_guide['fgmin'] <= recipe['FG'] <= style_guide['fgmax']
            abv_compliant = style_guide['abvmin'] <= recipe['ABV'] <= style_guide['abvmax']
            ibu_compliant = style_guide['ibumin'] <= recipe['IBU'] <= style_guide['ibumax']
            
            # Count total compliant metrics
            total_compliant = sum([og_compliant, fg_compliant, abv_compliant, ibu_compliant])
            
            results.append({
                'Style': recipe['Style'],
                'Total_Compliant': total_compliant,
                'OG_Compliant': og_compliant,
                'FG_Compliant': fg_compliant,
                'ABV_Compliant': abv_compliant,
                'IBU_Compliant': ibu_compliant
            })
        
        return pd.DataFrame(results)
    
    # Second part: Create summary table
    # Create compliance DataFrame
    compliance_df = check_recipe_compliance(data, data2)
    
    # Create summary table
    summary_table = pd.DataFrame()
    
    # Calculate metrics per style
    for style in compliance_df['Style'].unique():
        style_data = compliance_df[compliance_df['Style'] == style]
        total_recipes = len(style_data)
        
        # Calculate percentages for compliance levels (cumulative)
        compliance_counts = pd.Series({
            'Total_Recipes': total_recipes,
            'Met 4 of 4 (%)': np.round((sum(style_data['Total_Compliant'] == 4) / total_recipes * 100), 1),
            'Met 3 of 4 (%)': np.round((sum(style_data['Total_Compliant'] >= 3) / total_recipes * 100), 1),
            'Met 2 of 4 (%)': np.round((sum(style_data['Total_Compliant'] >= 2) / total_recipes * 100), 1),
            'Met 1 of 4 (%)': np.round((sum(style_data['Total_Compliant'] >= 1) / total_recipes * 100), 1),
            'Met Zero (%)': np.round((sum(style_data['Total_Compliant'] == 0) / total_recipes * 100), 1),
            'OG Compliant (%)': np.round((sum(style_data['OG_Compliant']) / total_recipes * 100), 1),
            'FG Compliant (%)': np.round((sum(style_data['FG_Compliant']) / total_recipes * 100), 1),
            'ABV Compliant (%)': np.round((sum(style_data['ABV_Compliant']) / total_recipes * 100), 1),
            'IBU Compliant (%)': np.round((sum(style_data['IBU_Compliant']) / total_recipes * 100), 1)
        })
        
        summary_table = pd.concat([summary_table, pd.DataFrame(compliance_counts).T], axis=0)
    
    summary_table.index = compliance_df['Style'].unique()
    
    # Sort the summary table by 'Met 4 of 4 (%)' in descending order
    summary_table_sorted = summary_table.sort_values('Met 4 of 4 (%)', ascending=False)
    
    # Third part: Display the table
    console = Console()
    table = Table(show_header=True, header_style="bold white")
    
    # Add columns with adjusted Style column width
    table.add_column("Style", style="bold white", justify="left", width=33)
    table.add_column("Total Recipes", justify="right", style="white")
    table.add_column("Met 4 of 4 (%)", justify="right", style="bold white")
    table.add_column("Met 3 of 4 (%)", justify="right", style="white")
    table.add_column("Met 2 of 4 (%)", justify="right", style="white")
    table.add_column("Met 1 of 4 (%)", justify="right", style="white")
    table.add_column("Met Zero (%)", justify="right", style="white")
    table.add_column("OG (%)", justify="right", style="white")
    table.add_column("FG (%)", justify="right", style="white")
    table.add_column("ABV (%)", justify="right", style="white")
    table.add_column("IBU (%)", justify="right", style="white")

    # Add rows with integers instead of decimals
    for style in summary_table_sorted.index:
        row = summary_table_sorted.loc[style]
        table.add_row(
            str(style),
            str(int(row['Total_Recipes'])),
            f"{int(row['Met 4 of 4 (%)'])}",
            f"{int(row['Met 3 of 4 (%)'])}",
            f"{int(row['Met 2 of 4 (%)'])}",
            f"{int(row['Met 1 of 4 (%)'])}",
            f"{int(row['Met Zero (%)'])}",
            f"{int(row['OG Compliant (%)'])}",
            f"{int(row['FG Compliant (%)'])}",
            f"{int(row['ABV Compliant (%)'])}",
            f"{int(row['IBU Compliant (%)'])}"
        )

    # Set table width
    table.width = 115

    # Print the table
    console.print(table)
    
    # Return the summary table for further use if needed
    return summary_table_sorted

# Usage example:
# summary_table = create_and_display_compliance_summary(beer_recipes, bjcp_style)

Visualization 7: Heat map of the adherence to style for the top 30 Recipes.

In [ ]:
# Create a heatmap of the compliance of the recipes to the Style Guidelines for top 20 Styles
def create_compliance_heatmap(summary_table, top_n=30):
    # Select columns for the heatmap
    heatmap_columns = [
        'Met 4 of 4 (%)', 'Met 3 of 4 (%)', 'Met 2 of 4 (%)', 
        'Met 1 of 4 (%)', 'OG Compliant (%)', 'FG Compliant (%)', 
        'ABV Compliant (%)', 'IBU Compliant (%)'
    ]
    
    # Sort by 'Met 4 of 4 (%)' and get top N styles
    plot_data = summary_table.sort_values('Total_Recipes', ascending=False).head(top_n)
    
    # Create the heatmap
    plt.figure(figsize=(12, 16))
    
    # Create heatmap
    sns.heatmap(plot_data[heatmap_columns], 
                annot=True,  # Show values in cells
                fmt='0.0f',  # Format as integer
                cmap='YlOrRd',  # Yellow-Orange-Red color scheme
                center=50,  # Center the colormap at 50%
                vmin=0,
                vmax=100,
                cbar_kws={'label': 'Percentage'})
    
    # Customize the plot
    plt.title('Beer Style Compliance Heatmap', pad=20)
    plt.xlabel('Compliance Metrics')
    plt.ylabel('Beer Styles')
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    plt.show()


### The call to run all of the visualizations.

In [ ]:
# Visulualization 1: Call the Horizontal bar chart of the frequency of recipes by Style
total_occurrence(beer_recipes)  # Pass the DataFrame as an argument

# Visulualization 2: Create the word cloud
create_beer_wordcloud(beer_recipes)

# Visulualization 3: The **Top 50** beer styles in a vertical bar chart
total_occurrence_abbrev_v(beer_recipes)

# Visulualization 4: Number of Recipes per Category
plot_category_counts(beer_recipes)

# Visualization 5: Call the Stacked Bar Chart 
plot_category_style_distribution(beer_recipes)  # Pass the DataFrame as an argument

#Visualization 6: Calculate and Call the Compliance Table
summary_table = create_and_display_compliance_summary(beer_recipes, bjcp_style)

#Visualization 7: Call the Compliance Heatmap
create_compliance_heatmap(summary_table)